In [17]:
# --- Install dependencies ---
!pip install ag2[openai] faiss-cpu

from autogen import AssistantAgent, UserProxyAgent
import faiss, numpy as np
from openai import OpenAI
import os

# --- Setup OpenAI Client ---
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---  Vector Memory Class ---
class VectorMemory:
    def __init__(self, dim=1536):  # dimension for text-embedding-3-small
        self.index = faiss.IndexFlatL2(dim)
        self.memories = []
        self.embeddings = []

    def embed(self, text):
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=[text]
        )
        return np.array(response.data[0].embedding, dtype='float32').reshape(1, -1)

    def add(self, text):
        emb = self.embed(text)
        self.index.add(emb)
        self.memories.append(text)
        self.embeddings.append(emb)

    def recall(self, query, k=3):
        query_emb = self.embed(query)
        D, I = self.index.search(query_emb, k)
        return [self.memories[i] for i in I[0]]

# --- Initialize Vector Memory ---
vector_memory = VectorMemory()

# --- Define Agents ---
assistant = AssistantAgent(name="assistant", llm_config={"model": "gpt-4"})
user_proxy = UserProxyAgent(name="user_proxy")
# Load key from Colab secrets
api_key = os.environ.get("OPENAI_API_KEY")

print(api_key[:8] + "..." )

# --- Register Memory Functions (AG2 style) ---

def add_to_memory(fact: str):
    """Store fact in vector memory"""
    vector_memory.add(fact)
    return f"Stored: {fact}"

def recall_memory(query: str):
    """Recall semantically similar memories"""
    results = vector_memory.recall(query)
    return f"Relevant past memories: {results}"

# Register functions by passing a dict
assistant.register_function({
    "add_to_memory": add_to_memory,
    "recall_memory": recall_memory
})


# Store facts
print(add_to_memory("AutoGen uses multiple agents to collaborate."))
print(add_to_memory("RAG improves answers by retrieving context from vector databases."))
print(add_to_memory("FAISS enables efficient similarity search."))

# Recall semantically similar memories
print(recall_memory("How does AutoGen use retrieval?"))

user_proxy.initiate_chat(
    assistant,
    message="Explain how AutoGen uses RAG with vector databases."
)



sk-proj-...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}